# Lesson 2 : LangGraph Components

In [11]:
from dotenv import load_dotenv
_ = load_dotenv()

> If you are not familiar with python typing annotation, you can refer to the [python documents](https://docs.python.org/3/library/typing.html).

In [52]:
from langchain.agents import tool
from langchain.tools.render import format_tool_to_openai_function
from langchain.prompts import ChatPromptTemplate
from langchain.schema.agent import AgentFinish
from langchain.agents.output_parsers import OpenAIFunctionsAgentOutputParser
from langchain_openai import ChatOpenAI

# Initialize GPT-4o
llm = ChatOpenAI(model_name="gpt-4o")



sys="""You are a helpful assistant with expertise in understanding and analyzing user queries. Your task is to analyze the given user query and determine which views (functions) should be called.



**Instructions:**  
1. Analyze the query and identify the intent of the user.
2. Determine which view(s) are relevant based on the query.
3. You can call:
    - **Only view_1** if the query is solely relevant to power outages.
    - **Only view_2** if the query is solely relevant to power bills.
    - **Only view_3** if the query is solely relevant to people's location.
    - **Return "null"** if the query is not relevant to any of the views.

You can call more than one view if the query involves multiple topics.

4. Provide the name of the relevant view(s) as the output, separated by commas if there are multiple.  
5. If the query is ambiguous, make a reasonable assumption based on context.

Now analyze the following user query and provide the appropriate view(s) to call:  
"{input}"
"""


# Define prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", sys ),
    ("user", "{input}"),
])

def route(result):
    if isinstance(result, AgentFinish):
        return result.return_values['output']
    else:
        tool_calls = result.additional_kwargs.get("tool_calls", [])
        tools = {
            "view_1": view_1, 
            "view_2": view_2,
        }
        responses = [tools[call.get("function").get("name")].run(call.get("function").get("arguments"))
                     for call in tool_calls]
        return responses

# Setup the chain
chain = prompt | llm | OpenAIFunctionsAgentOutputParser() 

result= chain.invoke({"input": "is data compliant"})

print(result.return_values['output'])



null


In [53]:
import re

output_text = result.return_values['output']

# Use regex to extract view names
views = re.findall(r'\bview_\d+\b', output_text)

print(views)

[]
